# E791 $D^+\to\pi^-\pi^+\pi^+$ — Fit 2 closure with multistart

This notebook duplicates the physical model of notebook 1 and performs a closure fit. The $\rho(770)$ coefficient is fixed to $1+0i$. All other complex `RealImag` coefficients float. Among dynamical parameters, only $m_{\rho(1450)}$ and $\Gamma_{\rho(1450)}$ float.

The E791 $D$ form factor is set to unity with `parent_radius=0.0`. The E791-to-project RBW sign convention is handled by shifting only the NR phase by $180^\circ$ while keeping $\rho(770)$ as the fixed reference.

The minimization uses multiple randomized starts and selects the lowest valid NLL without using the injected truth.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DecayChannel, DecayModel, Minimizer, NonResonant,
    Parameter, RealImag, Resonance, enable_x64, weighted_resample,
)

enable_x64()


## 1. Model and injected parameters


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

fit2_polar_published = {
    "sigma":   (1.17, 205.7),
    "rho770":  (1.00,   0.0),
    "NR":      (0.48,  57.3),
    "f0_980":  (0.43, 165.0),
    "f2_1270": (0.76,  57.3),
    "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def polar_to_xy(magnitude, phase_deg):
    phase = np.deg2rad(phase_deg)
    return magnitude*np.cos(phase), magnitude*np.sin(phase)

def internal_xy(name):
    magnitude, phase = fit2_polar_published[name]
    if name == "NR":
        phase += 180.0
    return polar_to_xy(magnitude, phase)

fit2_xy = {name: internal_xy(name) for name in fit2_polar_published}
truth = {}

def free_c(name):
    xt, yt = fit2_xy[name]
    truth[f"{name}.x"] = xt
    truth[f"{name}.y"] = yt
    return RealImag(
        Parameter.coefficient(f"{name}.x", 0.0, owner=name, bounds=(-2.0, 2.0), step=0.01),
        Parameter.coefficient(f"{name}.y", 0.0, owner=name, bounds=(-2.0, 2.0), step=0.01),
    )

rho1450_mass = Parameter.dynamics(
    "rho1450.mass", 1.45, owner="rho1450",
    bounds=(1.30, 1.60), step=0.002,
)
rho1450_width = Parameter.dynamics(
    "rho1450.width", 0.32, owner="rho1450",
    bounds=(0.15, 0.50), step=0.003,
)
truth["rho1450.mass"] = 1.465
truth["rho1450.width"] = 0.310

c = {
    "sigma": free_c("sigma"),
    "rho770": RealImag(1.0, 0.0),
    "NR": free_c("NR"),
    "f0_980": free_c("f0_980"),
    "f2_1270": free_c("f2_1270"),
    "f0_1370": free_c("f0_1370"),
    "rho1450": free_c("rho1450"),
}

components = [
    Resonance("sigma", (0,1), c["sigma"], mass=0.478, width=0.324, spin=0, resonance_radius=3.0, parent_radius=0.0),
    Resonance("rho770", (0,1), c["rho770"], mass=0.7693, width=0.1502, spin=1, resonance_radius=3.0, parent_radius=0.0),
    Resonance("f0_980", (0,1), c["f0_980"], mass=0.975, width=0.044, spin=0, resonance_radius=3.0, parent_radius=0.0),
    Resonance("f2_1270", (0,1), c["f2_1270"], mass=1.275, width=0.185, spin=2, resonance_radius=3.0, parent_radius=0.0),
    Resonance("f0_1370", (0,1), c["f0_1370"], mass=1.434, width=0.173, spin=0, resonance_radius=3.0, parent_radius=0.0),
    Resonance("rho1450", (0,1), c["rho1450"], mass=rho1450_mass, width=rho1450_width, spin=1, resonance_radius=3.0, parent_radius=0.0),
    NonResonant(c["NR"]),
]

model = DecayModel(channel, components)
print("free parameters:")
for p in model.parameters:
    if not p.fixed:
        print(f"  {p.name:14s} bounds={p.bounds}")


## 2. Pseudo-data and normalization sample


In [ ]:
N_POOL = 1_000_000
N_DATA = 100_000
N_NORM = 1_000_000

pool = model.generate_phase_space(N_POOL, seed=2000)
target_w = pool.weights * model.intensity(pool.as_dict(), truth)
print("finite target weights:", bool(jnp.all(jnp.isfinite(target_w))))
print("positive total target weight:", float(jnp.sum(target_w)) > 0.0)

data = weighted_resample(
    jax.random.key(791), pool, target_w, N_DATA, replace=True
)
norm = model.generate_phase_space(N_NORM, seed=2027)
cache = model.prepare_cache(data, norm)
print("data:", data.size, "normalization:", norm.size)


In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
h = ax.hist2d(np.asarray(data.s12), np.asarray(data.s13), bins=100)
fig.colorbar(h[3], ax=ax, label="events")
ax.set(xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]", title="E791 Fit 2-based pseudo-data")
plt.show()


## 3. Unbinned NLL and multistart minimization

The truth is printed only as a closure diagnostic. It is not used as a start or in the selection of the best minimum.


In [ ]:
def nll(values):
    intensity, normalization = cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300))) + data.size*jnp.log(normalization)

minimizer = Minimizer(nll, model.parameters, tolerance=1e-6)
print("NLL(truth):", float(nll(truth)))

scan = minimizer.fit_multistart(
    n_starts=20,
    seed=314159,
    include_default=False,
    simplex=True,
)
result = scan.best
fit_values = {name: float(result.values[name]) for name in result.parameters}

print("best valid:", result.valid)
print("best NLL:", float(result.fval))
print("NLL(best)-NLL(truth):", float(result.fval - nll(truth)))
print("best EDM:", float(result.fmin.edm))


### Scan diagnostics


In [ ]:
print(f"{'start':>5s} {'valid':>7s} {'NLL':>16s} {'EDM':>12s}")
for i, trial in enumerate(scan.results):
    print(
        f"{i:5d} {str(bool(trial.valid)):>7s} "
        f"{float(trial.fval):16.6f} {float(trial.fmin.edm):12.4e}"
    )


## 4. Closure table

The requested compatibility criterion is $|(\theta_{gen}-\theta_{fit})/\sigma_{fit}|<1$.


In [ ]:
print(f"{'parameter':14s} {'gen':>10s} {'fit':>10s} {'err':>10s} {'pull':>9s} {'<1sigma':>9s}")
for p in model.parameters:
    if p.fixed:
        continue
    gen = truth[p.name]
    fit = float(result.values[p.name])
    err = float(result.errors[p.name])
    pull = (gen-fit)/err
    print(f"{p.name:14s} {gen:10.5f} {fit:10.5f} {err:10.5f} {pull:9.3f} {str(abs(pull)<1):>9s}")


## 5. Projection at a representative random start and after the fit


In [ ]:
display_start = scan.starts[0]

def proj(values, bins):
    w = np.asarray(norm.weights * model.intensity(norm.as_dict(), values))
    h12, _ = np.histogram(np.asarray(norm.s12), bins=bins, weights=w)
    h13, _ = np.histogram(np.asarray(norm.s13), bins=bins, weights=w)
    return h12 + h13

s = np.concatenate([np.asarray(data.s12), np.asarray(data.s13)])
bins = np.linspace(s.min(), s.max(), 110)
centers = 0.5*(bins[:-1]+bins[1:])
hd, _ = np.histogram(s, bins=bins)
hs = proj(display_start, bins)
hf = proj(fit_values, bins)
ht = proj(truth, bins)
for h in (hs, hf, ht):
    h *= hd.sum()/h.sum()

fig, ax = plt.subplots(figsize=(10,5.5))
ax.errorbar(centers, hd, yerr=np.sqrt(np.maximum(hd,1)), fmt=".", label="pseudo-data")
ax.step(centers, hs, where="mid", label="one random start")
ax.step(centers, hf, where="mid", label="best multistart fit")
ax.step(centers, ht, where="mid", linestyle="--", label="generated model")
ax.set(xlabel=r"$m^2(\pi^-\pi^+)$ [GeV$^2$]", ylabel="entries / bin", title="E791 Fit 2-based projection")
ax.legend()
plt.show()

mask = (centers > 1.30**2) & (centers < 1.60**2)
fig, ax = plt.subplots(figsize=(9,5))
ax.errorbar(centers[mask], hd[mask], yerr=np.sqrt(np.maximum(hd[mask],1)), fmt=".", label="pseudo-data")
ax.step(centers[mask], hs[mask], where="mid", label="one random start")
ax.step(centers[mask], hf[mask], where="mid", label="best multistart fit")
ax.step(centers[mask], ht[mask], where="mid", linestyle="--", label="generated model")
ax.set(xlabel=r"$m^2(\pi^-\pi^+)$ [GeV$^2$]", ylabel="entries / bin", title=r"$\rho(1450)$-sensitive region")
ax.legend()
plt.show()
